# Experimental iteration speed (D5)

Time to one calibrated 90% prediction interval, end to end, from raw data.
Protocol: `iteration_speed_d5/PROTOCOL.md`. Results: `iteration_speed_d5/results.json`.

In [1]:
import json
from pathlib import Path

import pandas as pd

data = json.loads((Path("iteration_speed_d5") / "results.json").read_text())
table = pd.DataFrame(data["records"])
wide = table.pivot(index="n", columns="method", values="median_seconds").round(3)
wide

method,catboost_cqr,lightgbm_cqr,mboostrs_cqr,mboostrs_discover
n,,,,
1000,0.208,0.934,0.250,0.075
10000,0.448,1.023,2.971,0.857


## Gap to the best competitor

The best competitor at each size is the faster of CatBoost CQR and LightGBM CQR.
Positive means slower than that competitor.

In [2]:
rows = []
for n, r in wide.iterrows():
    best = min(["catboost_cqr", "lightgbm_cqr"], key=lambda k: r[k])
    for method in ["mboostrs_cqr", "mboostrs_discover"]:
        rows.append({"n": n, "method": method, "best competitor": best,
                     "gap to best (%)": round(100 * (r[method] / r[best] - 1))})
pd.DataFrame(rows)

,n,method,best competitor,gap to best (%)
0,1000,mboostrs_cqr,catboost_cqr,20
1,1000,mboostrs_discover,catboost_cqr,-64
2,10000,mboostrs_cqr,catboost_cqr,563
3,10000,mboostrs_discover,catboost_cqr,91


## Reading the gaps

- **n = 1,000.** mboostrs discover is faster than both competitors. mboostrs CQR
  is within about a quarter of CatBoost CQR.
- **n = 10,000.** mboostrs CQR takes about 7 times CatBoost CQR's time. This gap
  is filed as an issue; the step search of the quantile fits is the likely cause.
- **LightGBM at n = 1,000** is dominated by a fixed startup cost (see the protocol).

Three repeats on one machine. Treat the gaps as indicative until the CQR step
search is profiled.

(where-the-cqr-time-goes-n--10000)=
## CQR profiling (n = 10,000)

One CQR interval at n = 10,000 was profiled once (seed 0). Its quantile fits run
7 refits and 7 cross-validations, as the tuning grid doubles. The cross-validation
takes about 60% of the time and the refits about 40%. Starting the grid at 1,600
steps instead of 200 gives a similar time (3.2 s against 3.2 s on three seeds).
Source: `iteration_speed_d5/cqr_profile_results.json`.

In [3]:
import json

profile = json.loads((Path("iteration_speed_d5") / "cqr_profile_results.json").read_text())
floor = pd.DataFrame({
    "step floor": ["200", "1600"],
    "median seconds (n = 10,000, 3 seeds)": [round(sorted(profile["floor_timing_n10000"]["200"])[1], 2),
                                              round(sorted(profile["floor_timing_n10000"]["1600"])[1], 2)],
})
floor

,step floor,"median seconds (n = 10,000, 3 seeds)"
0,200,3.23
1,1600,3.16
